# CRUD Operations in ChromaDB (Adding and Querying Documents)

**CRUD** = **C**reate, **R**ead, **U**pdate, **D**elete

### What you will learn
1. Create a client and collection
2. Add documents with ids and metadata
3. Read with `peek`, `get`, and `query`, plus filters
4. Update and upsert
5. Delete by id and by filter
6. Check the collection state

In [ ]:
!pip install -q chromadb

## 1. CREATE: Client and Collection

In [ ]:
import chromadb

client = chromadb.Client()

# Delete a previous run's collection (if any) so this cell is re-runnable
try:
    client.delete_collection("movies_demo")
except Exception:
    pass

collection = client.create_collection(name="movies_demo")
print("Collections:", [getattr(c, "name", c) for c in client.list_collections()])
print("Document count:", collection.count())

## 2. CREATE: Add Documents
Each document needs a unique `id`. Adding a duplicate `id` raises an error or is ignored, so use `upsert()` when unsure.

In [ ]:
collection.add(
    documents=[
        "Interstellar follows astronauts travelling through a wormhole to save humanity.",
        "The Lion King is an animated story of a young lion prince.",
        "Inception is a thriller about stealing secrets from people's dreams.",
        "Chennai Express is a comedy about a train journey to Tamil Nadu.",
        "Toy Story is an animated film about toys that come to life.",
    ],
    metadatas=[
        {"genre": "scifi",     "year": 2014},
        {"genre": "animation", "year": 1994},
        {"genre": "scifi",     "year": 2010},
        {"genre": "comedy",    "year": 2013},
        {"genre": "animation", "year": 1995},
    ],
    ids=["m1", "m2", "m3", "m4", "m5"],
)
print("Added. Count:", collection.count())

## 3. READ: Peek

In [ ]:
preview = collection.peek(limit=3)
for i, d in zip(preview["ids"], preview["documents"]):
    print(i, "->", d)

## 4. READ: Get by ID (exact lookup, no similarity)

In [ ]:
res = collection.get(ids=["m1", "m3"])
for i, d, m in zip(res["ids"], res["documents"], res["metadatas"]):
    print(f"{i} | {m} | {d}")

`get()` can also filter by metadata without any similarity search:

In [ ]:
res = collection.get(where={"genre": "animation"})
print(res["ids"])

## 5. READ: Semantic Query

In [ ]:
def show(results):
    for i, d, dist in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
        print(f"{i} | distance={dist:.4f} | {d}")
    print()

show(collection.query(query_texts=["a film about space travel"], n_results=2))

## 6. READ: Query with Filters

In [ ]:
# Equality filter
show(collection.query(query_texts=["a fun film for kids"], n_results=2, where={"genre": "animation"}))

# Comparison filter ($gt = greater than)
show(collection.query(query_texts=["a good movie"], n_results=3, where={"year": {"$gt": 2005}}))

# Combined filter with $and
show(collection.query(
    query_texts=["a good movie"], n_results=3,
    where={"$and": [{"genre": "scifi"}, {"year": {"$gte": 2012}}]},
))

## 7. UPDATE

In [ ]:
collection.update(
    ids=["m4"],
    documents=["Chennai Express is a romantic comedy about a train journey from Mumbai to Rameswaram."],
    metadatas=[{"genre": "comedy", "year": 2013, "updated": True}],
)
print(collection.get(ids=["m4"]))

## 8. UPSERT (add or update)

In [ ]:
collection.upsert(
    documents=[
        "Inception is a mind-bending thriller about entering dreams to plant an idea.",  # existing id -> updated
        "Vikram Vedha is a Tamil crime thriller about a cop and a gangster.",            # new id -> created
    ],
    metadatas=[{"genre": "scifi", "year": 2010}, {"genre": "thriller", "year": 2017}],
    ids=["m3", "m6"],
)
print("Count after upsert:", collection.count())
print(collection.get(ids=["m3", "m6"])["documents"])

## 9. DELETE by ID

In [ ]:
collection.delete(ids=["m5"])
print("Count:", collection.count(), "| ids:", collection.get()["ids"])

## 10. DELETE by Filter

In [ ]:
collection.delete(where={"genre": "animation"})
print("Count:", collection.count(), "| ids:", collection.get()["ids"])

## 11. Final State Check

In [ ]:
final = collection.get()
print("Final count:", collection.count())
for i, d, m in zip(final["ids"], final["documents"], final["metadatas"]):
    print(f"{i} | {m} | {d}")
print("Collections:", [getattr(c, "name", c) for c in client.list_collections()])

## 12. Practice Exercises (Solved)

In [ ]:
# Exercise 1: add 5 docs, upsert to update 2 and create 1 in a single call
ex = client.get_or_create_collection("exercise")
ex.add(documents=[f"Original note {n}" for n in range(1, 6)], ids=[f"n{n}" for n in range(1, 6)])

ex.upsert(
    documents=["Edited note 1", "Edited note 2", "Brand new note 6"],
    ids=["n1", "n2", "n6"],
)
print(ex.count(), ex.get()["documents"])

In [ ]:
# Exercise 2: delete all docs of one category and confirm
ex2 = client.get_or_create_collection("exercise2")
ex2.add(documents=["a", "b", "c", "d"], metadatas=[{"c": "x"}, {"c": "y"}, {"c": "x"}, {"c": "y"}], ids=["1", "2", "3", "4"])
ex2.delete(where={"c": "x"})
print("After delete:", ex2.count(), ex2.get()["ids"])

In [ ]:
# Exercise 3: safe_add never raises a duplicate-id error
def safe_add(collection, id, text, metadata=None):
    kwargs = {"ids": [id], "documents": [text]}
    if metadata:
        kwargs["metadatas"] = [metadata]
    collection.upsert(**kwargs)

safe_add(ex, "n1", "Safely overwritten note 1")
safe_add(ex, "n99", "Safely created note 99")
safe_add(ex, "n99", "Calling again with the same id is fine")
print(ex.get(ids=["n1", "n99"])["documents"])

## Key Takeaways

| Operation | Method | Notes |
|---|---|---|
| Create | `client.create_collection()` | Errors if the name already exists |
| Create / Add | `collection.add()` | Insert new documents |
| Read (exact) | `collection.get()` | By `id` or `where` filter |
| Read (semantic) | `collection.query()` | Closest matches plus distances |
| Read (preview) | `collection.peek()` | Quick sample |
| Update | `collection.update()` | Existing ids only |
| Add or Update | `collection.upsert()` | Creates or updates |
| Delete | `collection.delete()` | By `ids` or `where` filter |